# Trajectory Analysis for _Suo et. al._ iterative training

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
working_directory = "/home/icb/kemal.inecik/work/codes/sctram"
sys.path.append(working_directory)

import logging
import subprocess
import gc
import os
import time
import numpy as np
import pandas as pd
import networkx as nx
import scanpy as sc
import anndata as ad
import pickle
import itertools
import torch
import tqdm
import scvi

sc.settings.verbose = 3

In [3]:
from sctram.api._lower_level import TrajectoryEvaluationAPI
from sctram.generate.real import sc_suo_developmental_complete
from sctram.input import InputTrajectories

2025-03-07 00:20:52.098 | INFO     | sctram.api._defaults_read:load_default_metrics:23 - Loaded default metrics from /home/icb/kemal.inecik/work/codes/sctram/sctram/api/_defaults.yaml
2025-03-07 00:20:52.099 | INFO     | sctram.api._defaults_read:load_default_metrics:79 - Default metrics YAML structure validated successfully.


In [4]:
# Important to have consistent figures across platforms

%matplotlib inline
%config InlineBackend.figure_format='retina'

import pickle

from networkx.drawing.nx_agraph import graphviz_layout
from matplotlib import gridspec
import matplotlib.pyplot as plt
import seaborn as sns
import colorcet as cc
from adjustText import adjust_text  
import matplotlib.patheffects as path_effects

_rcparams_path = os.path.join(working_directory, "reproducibility/figure_rcparams/rcparams.pickle")
with open(_rcparams_path, "rb") as file:
    _rcparams = pickle.load(file)
plt.rcParams.update(_rcparams)

In [5]:
print(f"CUDA used: {torch.cuda.is_available()}")

dataset_dir = "/home/icb/kemal.inecik/lustre_workspace/temp_sctram_data"
helpers_directory = os.path.join(os.getcwd(), "helper")
logs_directory = os.path.join(os.getcwd(), "logs")

CUDA used: False


## Convert models to AnnData

In [6]:
epochs = list(itertools.chain(range(0, 19), range(19, 52, 3), range(52, 100, 6), range(100, 200, 10), range(200, 400, 20), range(400, 1001, 40)))
overwrite = False
print(f" - Number of models to be converted to AnnData: {len(epochs)!r}")

adata_suo = sc_suo_developmental_complete(dataset_dir=dataset_dir)
adata_file_path = os.path.join("/home/icb/kemal.inecik/lustre_workspace/tardis_data/processed", "dataset_complete_Suo.h5ad")
for epoch in epochs:
    for model_str in ["scanvi", "scvi"]:
        model_dir_path = os.path.join(dataset_dir, f"model_suo_incremental_training_{model_str}_epoch_{epoch}")
        output_dir_path = os.path.join(dataset_dir, f"adata_suo_incremental_training_{model_str}_epoch_{epoch}.h5ad")
        
        if overwrite or not os.path.exists(output_dir_path) or not os.path.isfile(output_dir_path):

            if not os.path.exists(model_dir_path) or not os.path.isdir(model_dir_path):
                print(f"Training is not prepared for model {model_str!r} and for epoch {epoch!r}.")
                continue
            
            adata = ad.read_h5ad(adata_file_path)
            assert np.all(adata_suo.obs.index == adata.obs.index)
            adata.obs = adata_suo.obs.copy()
            adata.obs["age"] = adata.obs["age"].astype("str").astype("category")
            if model_str == "scvi":
                vae = scvi.model.SCVI.load(model_dir_path, adata = adata)
            elif model_str == "scanvi":
                vae = scvi.model.SCANVI.load(model_dir_path, adata = adata)
            latent = ad.AnnData(X=vae.get_latent_representation(), obs=adata.obs.copy())
    
            latent.write_h5ad(output_dir_path)
            del adata, vae, latent
            gc.collect()
            print(f"AnnData prepared for model {model_str!r} and for epoch {epoch!r}.")
print(" - Completed.")     

2025-03-07 00:20:52.849 | WARNING  | sctram.generate.real._download:download_dataset:105 - File PosixPath('/home/icb/kemal.inecik/lustre_workspace/temp_sctram_data/suo_developmental_complete.h5ad') already exists. Skipping download.


 - Number of models to be converted to AnnData: 74
 - Completed.


# Running the `sctram` package

In [7]:
input_trajectories_path_litc_1_name = f"adata_suo_input_haematopoeitic_lineage_litc_1.pkl"
input_trajectories_path_litc_2_name = f"adata_suo_input_haematopoeitic_lineage_litc_2.pkl"
input_trajectories_path_name = f"adata_suo_input_haematopoeitic_lineage.pkl"

itpn_list = [input_trajectories_path_name] #, input_trajectories_path_litc_1_name, input_trajectories_path_litc_2_name]

override = False
lineage = "Haematopoeitic_lineage"
count = 0
lineage_part = lineage.replace("_lineage", "").lower()

for epoch in epochs:
    for use_rep in ["scanvi"]: #, "scvi"]:

        adata_path = os.path.join(dataset_dir, f"adata_suo_incremental_training_{use_rep}_epoch_{epoch}.h5ad")        
        if os.path.exists(adata_path) and os.path.isfile(adata_path):
    
            for itpn in itpn_list:
                itpn_base = os.path.splitext(itpn)[0]
                itp = os.path.join(dataset_dir, itpn)
                with open(itp, "rb") as _file:
                    litc = pickle.load(_file)
                
                for trajectory in sorted(litc.graph["trajectories"]):
                    output_file = os.path.join(dataset_dir, f"_metric_adata_suo_iterative_{lineage_part}_{use_rep}_{epoch}_{itpn_base}_{trajectory}.pickle")
                    log_file = os.path.join(logs_directory, f"slurm_out_metric_adata_suo_iterative_{lineage_part}_{use_rep}_{epoch}_{itpn_base}_{trajectory}.log")
                    slurmjob_name = f"sctram_iter_{use_rep}_{epoch}"

                    if override or not os.path.exists(output_file) or not os.path.isfile(output_file):
                        try:
                            slurm_script = f"""#!/bin/bash
#SBATCH -J {slurmjob_name}
#SBATCH -p cpu_p
#SBATCH --qos cpu_normal
#SBATCH -c 32
#SBATCH --mem=293G
#SBATCH --nice=0
#SBATCH -t 11:50:00
#SBATCH -o {log_file}
#SBATCH -e {log_file}

source activate sctram_dev_env
python -u {os.path.join(helpers_directory, 'suo_sctram_iterative.py')} --lineage "{lineage}" --use_rep "{use_rep}" --epoch "{epoch}" --itpn_base "{itpn_base}" --trajectory "{trajectory}"
            """
                            script_name = os.path.join(logs_directory, f"slurm_job_metric_adata_suo_iterative_{lineage_part}_{use_rep}_{epoch}_{itpn_base}_{trajectory}.sh")
                            with open(script_name, "w") as f:
                                f.write(slurm_script)
            
                            print(f"Submitted job {count+1!r} of {lineage!r} with {use_rep!r} of trajectory {trajectory!r} for epoch {epoch!r} for {itpn_base!r} of {trajectory!r}")
                            subprocess.run(["sbatch", script_name])
                            count += 1
                            # if count > 1:
                            #     raise ValueError
                        finally:
                            time.sleep(0.1)
                            os.remove(script_name)

                    else:
                        print(f"Already exist: {lineage!r} with {use_rep!r} of trajectory {trajectory!r} for epoch {epoch!r} for {itpn_base!r} of {trajectory!r}")
            
        else:
            print(f"AnnData is not prepared for model {use_rep!r} and for epoch {epoch!r}.")

print(f" - Number of jobs submitted: {count}")

Submitted job 1 of 'Haematopoeitic_lineage' with 'scanvi' of trajectory 'alternative_myeloid' for epoch 0 for 'adata_suo_input_haematopoeitic_lineage' of 'alternative_myeloid'
Submitted batch job 33788379
Submitted job 2 of 'Haematopoeitic_lineage' with 'scanvi' of trajectory 'b_cell_specialization' for epoch 0 for 'adata_suo_input_haematopoeitic_lineage' of 'b_cell_specialization'
Submitted batch job 33788380
Submitted job 3 of 'Haematopoeitic_lineage' with 'scanvi' of trajectory 'b_cells' for epoch 0 for 'adata_suo_input_haematopoeitic_lineage' of 'b_cells'
Submitted batch job 33788381
Submitted job 4 of 'Haematopoeitic_lineage' with 'scanvi' of trajectory 'cd4' for epoch 0 for 'adata_suo_input_haematopoeitic_lineage' of 'cd4'
Submitted batch job 33788382
Submitted job 5 of 'Haematopoeitic_lineage' with 'scanvi' of trajectory 'complete_stem_trajectory' for epoch 0 for 'adata_suo_input_haematopoeitic_lineage' of 'complete_stem_trajectory'
Submitted batch job 33788383
Submitted job 6 o